In [2]:
import dask.dataframe as dd
import locale
import os
import numpy as np
import pandas as pd
import shutil

In [4]:
# défintion de paramètres locaux fr pour obtenir les mois et jour en français
locale.setlocale(locale.LC_ALL, 'fr_FR.utf8')

'fr_FR.utf8'

In [5]:
# Création d'un dictionnaire des dtypes
dic_dtypes = {'Viaje id' : "int32",
          'Usuario_Id' : "int32",
          'Genero' : "object",
          'Año_de_nacimiento' : "float64",
          'Inicio_del_viaje' : "object",
          'Fin_del_viaje' : "object",
          'Origen_Id' : "int32",
          'Destino_Id' : "int32",
          "id" : "int32",
          "name" : "object", 
          "obcn" : "object", 
          "location" : "category",
          "latitude" : "float32",
          "longitude" : "float32",
          "status" : "object"}

In [50]:
# Charger le fichier nomenclature qui servira à la jointure et obtenir le nom des stations et leur localisation à partir de leur id
df_stations = pd.read_csv('data/nomenclatura_2025_11.csv', dtype=dic_dtypes)

# Renommer la colonne 'id' en 'station_id' pour la clarté et la jointure
df_stations = df_stations.rename(columns={'id': 'station_id', 'name':'nom', 'location':'lieu'})

# Sélectionner uniquement les colonnes nécessaires (pour éviter le bruit)
cols_a_garder = ['station_id', 'nom', 'lieu', 'latitude', 'longitude']
df_stations = df_stations[cols_a_garder]

In [51]:
df_stations.head()

,station_id,nom,lieu,latitude,longitude
0,2,(GDL-001) C. Epigmenio Glez./ Av. 16 de Sept.,POLÍGONO CENTRAL,20.666378,-103.348824
1,3,(GDL-002) C. Colonias / Av. Niños héroes,POLÍGONO CENTRAL,20.667229,-103.365997
2,4,(GDL-003) C. Vidrio / Av. Chapultepec,POLÍGONO CENTRAL,20.667690,-103.368256
3,5,(GDL-004) C. Ghilardi /C. Miraflores,POLÍGONO CENTRAL,20.691847,-103.362549
4,6,(GDL-005) C. San Diego /Calzada Independencia,POLÍGONO CENTRAL,20.681158,-103.339363


In [6]:
all_files = 'data/201*/*.csv'

ddf = dd.read_csv(
    all_files,
    sep=',',
    encoding='ISO-8859-1',
    dtype=dic_dtypes,
    assume_missing=True # Pour être sûr que les colonnes int passent en float si besoin
)

# Renommer les colonnes
ddf.columns = ["id_trajet", "id_utilisateur", "genre", "annee_naissance", "debut_trajet", "fin_trajet", "id_depart", "id_arrivee"]

# Afficher les métadonnées pour confirmation
print("--- Dask DataFrame chargé ---")
print(ddf.head())
print(ddf.dtypes)

--- Dask DataFrame chargé ---
   id_trajet  id_utilisateur genre  annee_naissance         debut_trajet  \
0    42034.0           17636     M           1989.0  2015-01-01 00:03:06   
1    42046.0           10701     M           1986.0  2015-01-01 06:00:24   
2    42047.0           13525     M           1983.0  2015-01-01 07:06:08   
3    42048.0           17625     M           1980.0  2015-01-01 07:25:14   
4    42049.0             272     M           1980.0  2015-01-01 07:36:56   

            fin_trajet  id_depart  id_arrivee  
0  2015-01-01 00:05:52         19          19  
1  2015-01-01 06:03:53         37          25  
2  2015-01-01 07:06:59         55          55  
3  2015-01-01 07:33:13         81          54  
4  2015-01-01 07:48:33         79           9  
id_trajet                  float64
id_utilisateur               int32
genre              string[pyarrow]
annee_naissance            float64
debut_trajet       string[pyarrow]
fin_trajet         string[pyarrow]
id_depart      

Récupérer les données des stations pour les joindre aux données

In [53]:
# Mettre en place la première jointure sur l'id de départ
ddf = ddf.merge(
    df_stations,
    left_on='id_depart',  # Clé de jointure dans le DataFrame des trajets
    right_on='station_id', # Clé de jointure dans le DataFrame des stations
    how='left',           # Conserver tous les trajets même si l'ID de station est manquant
    suffixes=('', '_depart') # Ajoute le suffixe '_depart' aux colonnes de stations
)

# Renommer les colonnes de coordonnées et de nom pour le départ
ddf = ddf.rename(columns={
    'nom': 'nom_station_depart',
    'lieu': 'quartier_depart',
    'latitude': 'lat_depart',
    'longitude': 'lon_depart'
})

# Supprimer la colonne station_id introduite par la jointure
ddf = ddf.drop(columns=['station_id'])


# Placer la colonne id_destination à la fin

ddf = ddf[[col for col in ddf.columns if col != 'id_arrivee'] + ['id_arrivee']]

# Mettre en place la seconde jointure sur l'id d'arrivée
ddf = ddf.merge(
    df_stations,
    left_on='id_arrivee',  # Clé de jointure dans le DataFrame des trajets
    right_on='station_id', # Clé de jointure dans le DataFrame des stations
    how='left',           # Conserver tous les trajets même si l'ID de station est manquant
    suffixes=('', '_arrivee') # Ajoute le suffixe '_arrivee' aux colonnes de stations
)

# Renommer les colonnes de coordonnées et de nom pour le départ
ddf = ddf.rename(columns={
    'nom': 'nom_station_arrivee',
    'lieu': 'quartier_arrivee',
    'latitude': 'lat_arrivee',
    'longitude': 'lon_arrivee'
})

# Supprimer la colonne station_id introduite par la jointure
ddf = ddf.drop(columns=['station_id'])



In [54]:
ddf.head()

,id_trajet,id_utilisateur,genre,annee_naissance,debut_trajet,fin_trajet,id_depart,nom_station_depart,quartier_depart,lat_depart,lon_depart,id_arrivee,nom_station_arrivee,quartier_arrivee,lat_arrivee,lon_arrivee
0,42034.0,17636,M,1989.0,2015-01-01 00:03:06,2015-01-01 00:05:52,19,(GDL-017) Av. México /C. Bernardo de Balbuena,POLÍGONO CENTRAL,20.679087,-103.372841,19,(GDL-017) Av. México /C. Bernardo de Balbuena,POLÍGONO CENTRAL,20.679087,-103.372841
1,42046.0,10701,M,1986.0,2015-01-01 06:00:24,2015-01-01 06:03:53,37,(GDL-035) C. Mezquitán / Av. Hidalgo,POLÍGONO CENTRAL,20.677315,-103.353416,25,(GDL-023) C. Jesús / C. San Felipe,POLÍGONO CENTRAL,20.679310,-103.356277
2,42047.0,13525,M,1983.0,2015-01-01 07:06:08,2015-01-01 07:06:59,55,(GDL-053) C. Maestranza / Av. Juárez,POLÍGONO CENTRAL,20.675402,-103.345665,55,(GDL-053) C. Maestranza / Av. Juárez,POLÍGONO CENTRAL,20.675402,-103.345665
3,42048.0,17625,M,1980.0,2015-01-01 07:25:14,2015-01-01 07:33:13,81,(GDL-079) C. Donato Guerra /Av. La Paz,POLÍGONO CENTRAL,20.668863,-103.351303,54,(GDL-052) Av. Juárez / Av. 16 de Septiembre,POLÍGONO CENTRAL,20.675243,-103.347893
4,42049.0,272,M,1980.0,2015-01-01 07:36:56,2015-01-01 07:48:33,79,(GDL-077) C. Argentina / C. Pedro Moreno,POLÍGONO CENTRAL,20.675631,-103.361343,9,(GDL-007) C. Epigmenio Glez./Av. Cristobal C.,POLÍGONO CENTRAL,20.666771,-103.350563


In [55]:
# Utilisation de dd.to_datetime avec 'mixed' pour gérer les formats "MM/DD/YYYY" et "YYYY-MM-DD"
for col in ["debut_trajet", "fin_trajet"]:
    ddf[col] = dd.to_datetime(ddf[col], format='mixed', errors='coerce') 
    # 'errors='coerce' convertira les dates non valides en NaT (Not a Time)

In [56]:
# La conversion des dates transforment ces colonnes en float au lieu de int, on applique un correctif
cols_to_fix = ["id_trajet", "id_utilisateur", "id_depart", "id_arrivee"]

for col in cols_to_fix:
    ddf[col] = ddf[col].astype('int32')

In [57]:
# Supprimer les lignes où la date de début de trajet est manquante
ddf = ddf.dropna(subset=['debut_trajet']) 
# Supprimer les lignes où l'âge de naissance est manquant
ddf = ddf.dropna(subset=['annee_naissance'])
# Supprimer les lignes où les infos de stations sont manquantes
ddf = ddf.dropna(subset=['nom_station_depart'])
ddf = ddf.dropna(subset=['nom_station_arrivee'])


# Conversion de l'année de naissance en entier (puisque les NA sont nettoyés)
# Nous devons utiliser un entier qui supporte les NA (Int64 de Pandas)
ddf['annee_naissance'] = ddf['annee_naissance'].astype('Int64')

print("\n--- Dask DataFrame après nettoyage (lazy) ---")
print(ddf.head())
print(ddf.dtypes)


--- Dask DataFrame après nettoyage (lazy) ---
   id_trajet  id_utilisateur genre  annee_naissance        debut_trajet  \
0      42034           17636     M             1989 2015-01-01 00:03:06   
1      42046           10701     M             1986 2015-01-01 06:00:24   
2      42047           13525     M             1983 2015-01-01 07:06:08   
3      42048           17625     M             1980 2015-01-01 07:25:14   
4      42049             272     M             1980 2015-01-01 07:36:56   

           fin_trajet  id_depart  \
0 2015-01-01 00:05:52         19   
1 2015-01-01 06:03:53         37   
2 2015-01-01 07:06:59         55   
3 2015-01-01 07:33:13         81   
4 2015-01-01 07:48:33         79   

                              nom_station_depart   quartier_depart  \
0  (GDL-017) Av. México /C. Bernardo de Balbuena  POLÍGONO CENTRAL   
1           (GDL-035) C. Mezquitán / Av. Hidalgo  POLÍGONO CENTRAL   
2           (GDL-053) C. Maestranza / Av. Juárez  POLÍGONO CENTRAL   
3    

In [58]:
# Ajout d'une colonne année (nécessaire pour le calcul de l'âge)
ddf['annee'] = ddf.debut_trajet.dt.year

# Ajout d'une colonne âge
ddf['age'] = ddf["annee"] - ddf["annee_naissance"]

# Ajout d'une colonne temps de trajet convertie en minutes
ddf['temps_trajet'] = (ddf.fin_trajet - ddf.debut_trajet) / pd.Timedelta(minutes=1)

# Ajout d'une colonne heure de départ
ddf['heure_debut'] = ddf.debut_trajet.dt.hour
ddf['heure_debut'] = ddf['heure_debut'].replace(0, 24) # On remplace 0 par 24 pour minuit pour l'affichage des graphiques

# Ajout d'une colonne mois et jour (pour l'analyse temporelle)
ddf['mois'] = ddf.debut_trajet.dt.strftime('%B')
ddf['jour'] = ddf.debut_trajet.dt.strftime('%A')

print("\n--- Dask DataFrame après ajout de colonnes (lazy) ---")
print(ddf.head())
print(ddf.dtypes)


--- Dask DataFrame après ajout de colonnes (lazy) ---
   id_trajet  id_utilisateur genre  annee_naissance        debut_trajet  \
0      42034           17636     M             1989 2015-01-01 00:03:06   
1      42046           10701     M             1986 2015-01-01 06:00:24   
2      42047           13525     M             1983 2015-01-01 07:06:08   
3      42048           17625     M             1980 2015-01-01 07:25:14   
4      42049             272     M             1980 2015-01-01 07:36:56   

           fin_trajet  id_depart  \
0 2015-01-01 00:05:52         19   
1 2015-01-01 06:03:53         37   
2 2015-01-01 07:06:59         55   
3 2015-01-01 07:33:13         81   
4 2015-01-01 07:48:33         79   

                              nom_station_depart   quartier_depart  \
0  (GDL-017) Av. México /C. Bernardo de Balbuena  POLÍGONO CENTRAL   
1           (GDL-035) C. Mezquitán / Av. Hidalgo  POLÍGONO CENTRAL   
2           (GDL-053) C. Maestranza / Av. Juárez  POLÍGONO CENTRAL 

In [59]:
def haversine_vectorise(lon1, lat1, lon2, lat2):
    """
    Calcule la distance de Haversine entre deux jeux de coordonnées 
    (lon1, lat1) et (lon2, lat2). Les entrées sont des Séries Dask.
    Le résultat est en kilomètres (km).
    """
    # Rayon de la Terre en kilomètres (km)
    R = 6371 
    
    # Conversion des degrés en radians
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])

    # Différences de coordonnées
    dlon = lon2 - lon1
    dlat = lat2 - lat1

    # Formule de Haversine
    # Note: Dask utilise NumPy pour ces calculs vectoriels
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    
    # Distance finale
    distance_km = R * c
    return distance_km

# S'assurer que les coordonnées sont du type float pour le calcul
ddf['lat_depart'] = ddf['lat_depart'].astype('float64')
ddf['lon_depart'] = ddf['lon_depart'].astype('float64')
ddf['lat_arrivee'] = ddf['lat_arrivee'].astype('float64')
ddf['lon_arrivee'] = ddf['lon_arrivee'].astype('float64')


# Ajout de la colonne de distance de Haversine (en mode paresseux)
ddf['distance_km'] = haversine_vectorise(
    ddf['lon_depart'], 
    ddf['lat_depart'], 
    ddf['lon_arrivee'], 
    ddf['lat_arrivee']
)

print("--- Dask DataFrame après ajout de la colonne 'distance_km' ---")
print(ddf[['nom_station_depart', 'nom_station_arrivee', 'distance_km']].head())
print(ddf.dtypes['distance_km'])

--- Dask DataFrame après ajout de la colonne 'distance_km' ---
                              nom_station_depart  \
0  (GDL-017) Av. México /C. Bernardo de Balbuena   
1           (GDL-035) C. Mezquitán / Av. Hidalgo   
2           (GDL-053) C. Maestranza / Av. Juárez   
3        (GDL-079)  C. Donato Guerra /Av. La Paz   
4       (GDL-077) C. Argentina / C. Pedro Moreno   

                             nom_station_arrivee  distance_km  
0  (GDL-017) Av. México /C. Bernardo de Balbuena     0.000000  
1             (GDL-023) C. Jesús / C. San Felipe     0.371217  
2           (GDL-053) C. Maestranza / Av. Juárez     0.000000  
3    (GDL-052) Av. Juárez / Av. 16 de Septiembre     0.793206  
4  (GDL-007) C. Epigmenio Glez./Av. Cristobal C.     1.492776  
float64


In [60]:
# Suppression des données dupliquées
ddf= ddf.drop_duplicates(
    subset=['id_trajet'], 
    keep='first'
)

Appliquer des filtres pour supprimer les valeurs aberrantes

In [ ]:
# Filtre sur le temps de trajet (en minutes)
TEMPS_MIN_MINUTES = 1
TEMPS_MAX_MINUTES = 240 # 4 heures

ddf = ddf[
    (ddf['temps_trajet'] > TEMPS_MIN_MINUTES) & 
    (ddf['temps_trajet'] < TEMPS_MAX_MINUTES)
]

# Filtre sur la distance (en km)
# On ne filtre pas sur la distance minimale car si station_depart == station_arrivee, la distance est 0 malgré le trajet
DISTANCE_MAX_KM = 30 

ddf = ddf[
    (ddf['distance_km'] < DISTANCE_MAX_KM)
]

# Filtre sur les heures de service (il n'est possible d'emprunter un vélo qu'entre 5h et 0h59)
heure_debut_service = 5
heure_fin_service = 24

ddf = ddf[
    (ddf["heure_debut"] >= heure_debut_service) &
    (ddf["heure_debut"] <= heure_fin_service)
]

# Pas de filtre d'âge à cette étape, elle sera réalisée ultérieurement
# En effet, bien que certaines valeurs soient aberrantes, les trajets ont bien eu lieu et il serait dommage de les supprimer'''



"Pas de filtre d'âge à cette étape, elle sera réalisée ultérieurement\nEn effet, bien que certaines valeurs soient aberrantes, les trajets ont bien eu lieu et il serait dommage de les supprimer"

In [62]:
output_dir = 'cleaned_data_2015'

if os.path.exists(output_dir):
    shutil.rmtree(output_dir)
    print(f"Dossier '{output_dir}' supprimé.")
    
print("Déclenchement du calcul et écriture des données...")

# Utilisez compute=True par défaut pour que l'opération soit synchrone
ddf.to_parquet(
    path= output_dir,
    engine='pyarrow',  # Moteur de lecture/écriture recommandé pour la performance
    write_metadata_file=True, # Écrit le fichier _metadata pour des lectures Dask encore plus rapides
    compression='snappy' # Compression rapide et efficace
)

print(f"\n✅ Sauvegarde terminée. Les données sont disponibles dans le dossier")
print(f"Nombre de partitions écrites : {ddf.npartitions}")

Dossier 'cleaned_data_2015' supprimé.
Déclenchement du calcul et écriture des données...

✅ Sauvegarde terminée. Les données sont disponibles dans le dossier
Nombre de partitions écrites : 60
